# Random Forest Explainer

This notebook contains the machine-learning and data-preparation workflow behind an interactive Tableau visualization that explains how a Random Forest classifier makes decisions.

Using the **UCI Mushroom dataset**, the project trains a Random Forest classifier, examines the behaviour of individual decision trees, selects a set of visually interesting trees, and exports tree geometry, node logic, votes, and decision paths for Tableau.

The goal is not only to predict whether a mushroom is **edible or poisonous**, but to make the model's decision-making process visible and understandable.

**Tools:** Python · pandas · NumPy · scikit-learn · Matplotlib · Tableau

> The 50-tree forest is used as a candidate pool. Nine trees are then selected for the final visualization so that the Tableau view remains readable while still showing variation between trees.


## 1. Imports and data loading


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

DATA_URL = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/mushroom/agaricus-lepiota.data"
)

COLUMNS = [
    "target", "cap-shape", "cap-surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root", "stalk-surface-above-ring",
    "stalk-surface-below-ring", "stalk-color-above-ring",
    "stalk-color-below-ring", "veil-type", "veil-color", "ring-number",
    "ring-type", "spore-print-color", "population", "habitat"
]

df = pd.read_csv(DATA_URL, header=None, names=COLUMNS)

print(f"Rows: {len(df):,}")
print(df["target"].value_counts())
df.head()


## 2. Prepare features and target

The original dataset uses `e` for edible and `p` for poisonous.  
For the model:

- `1` = Edible
- `0` = Poisonous

All categorical predictors are one-hot encoded.


In [ ]:
y = df["target"].map({"p": 0, "e": 1})
X = pd.get_dummies(df.drop(columns=["target"]))

print("Encoded feature matrix:", X.shape)
print("Target classes:", sorted(y.unique()))


## 3. Train the candidate Random Forest

A 50-tree forest provides a larger pool of individual trees from which visually varied trees can be selected for the explainer. Tree depth is constrained so the resulting decision paths remain interpretable.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

candidate_forest = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    max_features="sqrt",
    bootstrap=True,
    random_state=RANDOM_STATE,
)

candidate_forest.fit(X_train, y_train)

train_accuracy = accuracy_score(
    y_train, candidate_forest.predict(X_train)
)
test_accuracy = accuracy_score(
    y_test, candidate_forest.predict(X_test)
)

print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Test accuracy:     {test_accuracy:.3f}")
print(f"Generalization gap: {train_accuracy - test_accuracy:.3f}")


### Individual-tree performance

A Random Forest combines many weaker decision trees. Looking at the trees separately helps show why the ensemble can be more reliable than any single tree.


In [ ]:
tree_performance = []

for tree_number, tree in enumerate(candidate_forest.estimators_, start=1):
    train_pred = tree.predict(X_train.values)
    test_pred = tree.predict(X_test.values)

    tree_performance.append({
        "Tree_Number": tree_number,
        "Train_Accuracy": accuracy_score(y_train, train_pred),
        "Test_Accuracy": accuracy_score(y_test, test_pred),
    })

df_tree_performance = pd.DataFrame(tree_performance)
df_tree_performance["Generalization_Gap"] = (
    df_tree_performance["Train_Accuracy"]
    - df_tree_performance["Test_Accuracy"]
)

df_tree_performance.head(10)


## 4. Helper functions for Tableau


In [ ]:
def extract_tableau_geometry(tree, feature_names, tree_name):
    """Create curved edge/node coordinates for drawing one decision tree in Tableau."""
    left = tree.tree_.children_left
    right = tree.tree_.children_right
    features = tree.tree_.feature
    values = tree.tree_.value
    samples = tree.tree_.n_node_samples

    positions = {}

    def calc_coords(node, depth, x_left, x_right):
        if node == -1:
            return

        x_mid = (x_left + x_right) / 2.0
        width = x_right - x_left

        x_jitter = np.sin(node * 2.17) * width * 0.18
        y_jitter = np.cos(node * 1.31) * 0.08

        positions[node] = (x_mid + x_jitter, depth + y_jitter)

        calc_coords(left[node], depth + 1, x_left, x_mid)
        calc_coords(right[node], depth + 1, x_mid, x_right)

    calc_coords(0, 0, 0.0, 1.0)

    records = []
    edge_id = 0
    root_samples = samples[0]

    def branch_size(n_samples):
        return 1 + 9 * np.sqrt(n_samples / root_samples)

    root_x, root_y = positions[0]
    trunk_points = 8
    trunk_length = 1.0

    for point_order in range(trunk_points):
        t = point_order / (trunk_points - 1)
        records.append({
            "Tree_Name": tree_name,
            "Path_ID": f"{tree_name}_TRUNK",
            "Point_Order": point_order,
            "Node_ID": -1,
            "X": root_x + np.sin(t * np.pi) * 0.015,
            "Y": root_y - trunk_length + t * trunk_length,
            "Label": "",
            "Type": "Trunk",
            "Is_Node": 0,
            "Samples": root_samples,
            "Branch_Size": 11 - 2 * t,
        })

    def traverse_edges(node):
        nonlocal edge_id

        if node == -1 or left[node] == -1:
            return

        for child in (left[node], right[node]):
            parent_x, parent_y = positions[node]
            child_x, child_y = positions[child]

            parent_feature = feature_names[features[node]].replace("_", ": ")

            n_points = 10
            t = np.linspace(0, 1, n_points)

            direction = np.sign(child_x - parent_x) or 1
            curve = 0.006 * np.sin(np.pi * t)

            x_points = (
                parent_x
                + (child_x - parent_x) * t
                + direction * curve
            )
            y_points = parent_y + (child_y - parent_y) * t

            size_parent = branch_size(samples[node])
            size_child = branch_size(samples[child])
            size_points = size_parent + (size_child - size_parent) * t

            for point_order in range(n_points):
                if point_order == 0:
                    label = parent_feature
                    node_type = "Split"
                    node_id = node
                    is_node = 1
                elif point_order == n_points - 1:
                    node_id = child
                    is_node = 1

                    if left[child] == -1:
                        label = (
                            "YES"
                            if values[child][0][1] > values[child][0][0]
                            else "NO"
                        )
                        node_type = label
                    else:
                        label = feature_names[features[child]].replace("_", ": ")
                        node_type = "Split"
                else:
                    label = ""
                    node_type = "Edge"
                    node_id = child
                    is_node = 0

                records.append({
                    "Tree_Name": tree_name,
                    "Path_ID": f"{tree_name}_E_{edge_id}",
                    "Point_Order": point_order,
                    "Node_ID": node_id,
                    "X": x_points[point_order],
                    "Y": y_points[point_order],
                    "Label": label,
                    "Type": node_type,
                    "Is_Node": is_node,
                    "Samples": samples[child],
                    "Branch_Size": size_points[point_order],
                })

            edge_id += 1
            traverse_edges(child)

    traverse_edges(0)
    return pd.DataFrame(records)


def extract_node_logic(tree, feature_names, tree_name):
    """Extract split logic, class counts, and node probabilities."""
    left = tree.tree_.children_left
    features = tree.tree_.feature
    thresholds = tree.tree_.threshold
    values = tree.tree_.value
    samples = tree.tree_.n_node_samples

    records = []

    for node_id in range(tree.tree_.node_count):
        is_leaf = left[node_id] == -1

        if is_leaf:
            feature = ""
            operator = ""
            threshold = np.nan
        else:
            feature = feature_names[features[node_id]]
            operator = "<="
            threshold = thresholds[node_id]

        value_poisonous = values[node_id][0][0]
        value_edible = values[node_id][0][1]
        total = value_poisonous + value_edible

        probability_poisonous = value_poisonous / total if total else 0
        probability_edible = value_edible / total if total else 0

        predicted_class = (
            "Edible" if value_edible > value_poisonous else "Poisonous"
        )

        records.append({
            "Tree_Name": tree_name,
            "Node_ID": node_id,
            "Is_Leaf": int(is_leaf),
            "Feature": feature,
            "Operator": operator,
            "Threshold": threshold,
            "Samples_Node": samples[node_id],
            "Value_Edible": value_edible,
            "Value_Poisonous": value_poisonous,
            "Probability_Edible": probability_edible,
            "Probability_Poisonous": probability_poisonous,
            "Predicted_Class": predicted_class,
        })

    return pd.DataFrame(records)


## 5. Select nine trees for the visualization

The original project used nine trees in the final Tableau view. The selected trees below are drawn from the 50-tree candidate forest.

The three `front_tree_numbers` are the trees given the strongest visual emphasis in the explainer.


In [ ]:
SELECTED_TREE_NUMBERS = [41, 23, 29, 13, 16, 17, 11, 6, 4]
FRONT_TREE_NUMBERS = [41, 23, 29]

selected_estimators = [
    candidate_forest.estimators_[tree_number - 1]
    for tree_number in SELECTED_TREE_NUMBERS
]

print("Selected source trees:", SELECTED_TREE_NUMBERS)


### Performance of the selected nine-tree ensemble

Because these nine trees are selected from a larger fitted forest, they are evaluated by combining their individual predictions rather than fitting a second, unrelated nine-tree model.


In [ ]:
def selected_forest_proba(X_data):
    probabilities = np.stack(
        [tree.predict_proba(X_data) for tree in selected_estimators],
        axis=0,
    )
    return probabilities.mean(axis=0)


def selected_forest_predict(X_data):
    proba = selected_forest_proba(X_data)
    return candidate_forest.classes_[np.argmax(proba, axis=1)]


selected_train_pred = selected_forest_predict(X_train.values)
selected_test_pred = selected_forest_predict(X_test.values)

selected_train_accuracy = accuracy_score(y_train, selected_train_pred)
selected_test_accuracy = accuracy_score(y_test, selected_test_pred)

print(f"Selected ensemble training accuracy: {selected_train_accuracy:.3f}")
print(f"Selected ensemble test accuracy:     {selected_test_accuracy:.3f}")
print(
    "Selected ensemble generalization gap: "
    f"{selected_train_accuracy - selected_test_accuracy:.3f}"
)


## 6. Export tree geometry and node logic for Tableau


In [ ]:
geometry_parts = []
logic_parts = []

for display_order, (source_tree_number, estimator) in enumerate(
    zip(SELECTED_TREE_NUMBERS, selected_estimators),
    start=1,
):
    tree_name = f"Tree {display_order}"

    geometry = extract_tableau_geometry(
        estimator,
        list(X.columns),
        tree_name,
    )
    geometry["Original_Tree_Number"] = source_tree_number
    geometry["Tree_Display_Order"] = display_order
    geometry["Is_Front_Tree"] = int(
        source_tree_number in FRONT_TREE_NUMBERS
    )

    logic = extract_node_logic(
        estimator,
        list(X.columns),
        tree_name,
    )
    logic["Original_Tree_Number"] = source_tree_number

    geometry_parts.append(geometry)
    logic_parts.append(logic)

df_geometry = pd.concat(geometry_parts, ignore_index=True)
df_logic = pd.concat(logic_parts, ignore_index=True)

df_tableau_trees = df_geometry.merge(
    df_logic,
    on=[
        "Tree_Name",
        "Node_ID",
        "Original_Tree_Number",
    ],
    how="left",
)

df_tableau_trees.to_csv(
    "random_forest_9_trees_with_logic.csv",
    index=False,
    sep=";",
)

print(
    "Exported random_forest_9_trees_with_logic.csv "
    f"({len(df_tableau_trees):,} rows)"
)


### Optional preview of the nine selected trees


In [ ]:
def preview_selected_trees():
    fig, axes = plt.subplots(3, 3, figsize=(18, 18))
    axes = axes.flatten()

    colors = {
        "Split": "#6A6A69",
        "YES": "#72b7b2",
        "NO": "#ff9da7",
    }

    for ax, display_order in zip(
        axes,
        range(1, len(SELECTED_TREE_NUMBERS) + 1),
    ):
        tree_name = f"Tree {display_order}"
        tree_data = df_geometry[
            df_geometry["Tree_Name"] == tree_name
        ]

        for _, group in tree_data.groupby("Path_ID"):
            group = group.sort_values("Point_Order")
            x = group["X"].to_numpy()
            y_values = group["Y"].to_numpy()
            sizes = group["Branch_Size"].to_numpy()

            for j in range(len(group) - 1):
                ax.plot(
                    x[j:j + 2],
                    y_values[j:j + 2],
                    linewidth=sizes[j],
                    color="0.45",
                    solid_capstyle="round",
                    zorder=1,
                )

        nodes = (
            tree_data[tree_data["Is_Node"] == 1]
            .drop_duplicates(subset=["Node_ID"])
        )

        for node_type, group in nodes.groupby("Type"):
            ax.scatter(
                group["X"],
                group["Y"],
                s=65,
                color=colors.get(node_type, "gray"),
                edgecolors="none",
                zorder=5,
            )

        source_number = SELECTED_TREE_NUMBERS[display_order - 1]
        ax.set_title(f"{tree_name} (source tree {source_number})")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# Uncomment to preview:
# preview_selected_trees()


## 7. Select mushrooms for the explanatory story

To make the explainer more useful than a set of generic examples, mushrooms are selected to represent different levels of agreement and disagreement between the nine chosen trees.


In [ ]:
def class_label(value):
    return "Edible" if value == 1 else "Poisonous"


all_probabilities = selected_forest_proba(X.values)
selected_predictions = selected_forest_predict(X.values)

class_to_col = {
    cls: i for i, cls in enumerate(candidate_forest.classes_)
}

story_pool = df.copy()
story_pool["Original_Index"] = story_pool.index
story_pool["Predicted_Class_Num"] = selected_predictions
story_pool["Predicted_Class"] = story_pool[
    "Predicted_Class_Num"
].map(class_label)

story_pool["Probability_Edible"] = all_probabilities[
    :, class_to_col[1]
]
story_pool["Probability_Poisonous"] = all_probabilities[
    :, class_to_col[0]
]

vote_records = []
path_signature_records = []

for original_index in X.index:
    x_one = X.loc[[original_index]].values
    edible_votes = 0
    poisonous_votes = 0
    paths = []

    for display_order, estimator in enumerate(
        selected_estimators,
        start=1,
    ):
        vote = estimator.predict(x_one)[0]
        vote_label = class_label(vote)

        edible_votes += vote_label == "Edible"
        poisonous_votes += vote_label == "Poisonous"

        node_path = estimator.decision_path(x_one).indices
        path_signature = "->".join(node_path.astype(str))
        paths.append(path_signature)

        path_signature_records.append({
            "Original_Index": original_index,
            "Tree_Name": f"Tree {display_order}",
            "Path": path_signature,
            "Tree_Vote": vote_label,
        })

    vote_records.append({
        "Original_Index": original_index,
        "Tree_Votes_Edible": edible_votes,
        "Tree_Votes_Poisonous": poisonous_votes,
        "Vote_Split": f"{edible_votes}:{poisonous_votes}",
        "Vote_Margin": abs(edible_votes - poisonous_votes),
        "Full_Path_Signature": " | ".join(paths),
    })

df_votes = pd.DataFrame(vote_records)
df_all_paths = pd.DataFrame(path_signature_records)

story_pool = story_pool.merge(
    df_votes,
    on="Original_Index",
)

unique_story_pool = story_pool.drop_duplicates(
    subset=["Full_Path_Signature"]
).copy()


In [ ]:
def pick_group(name, condition, n, sort_col, ascending=False):
    group = (
        unique_story_pool[condition]
        .sort_values(sort_col, ascending=ascending)
        .head(n)
        .copy()
    )
    group["Story_Group"] = name
    return group


N_TREES = len(selected_estimators)

selected_parts = [
    pick_group(
        "Clear edible: all trees agree",
        unique_story_pool["Tree_Votes_Edible"] == N_TREES,
        3,
        "Probability_Edible",
    ),
    pick_group(
        "Clear poisonous: all trees agree",
        unique_story_pool["Tree_Votes_Poisonous"] == N_TREES,
        3,
        "Probability_Poisonous",
    ),
    pick_group(
        "Almost edible: only 1 tree says poisonous",
        (
            unique_story_pool["Tree_Votes_Edible"] == N_TREES - 1
        )
        & (
            unique_story_pool["Tree_Votes_Poisonous"] == 1
        ),
        3,
        "Probability_Edible",
    ),
    pick_group(
        "Almost poisonous: only 1 tree says edible",
        (
            unique_story_pool["Tree_Votes_Poisonous"] == N_TREES - 1
        )
        & (
            unique_story_pool["Tree_Votes_Edible"] == 1
        ),
        3,
        "Probability_Poisonous",
    ),
    pick_group(
        "Mostly edible: 7 vs 2",
        (
            unique_story_pool["Tree_Votes_Edible"] == N_TREES - 2
        )
        & (
            unique_story_pool["Tree_Votes_Poisonous"] == 2
        ),
        2,
        "Probability_Edible",
    ),
    pick_group(
        "Mostly poisonous: 7 vs 2",
        (
            unique_story_pool["Tree_Votes_Poisonous"] == N_TREES - 2
        )
        & (
            unique_story_pool["Tree_Votes_Edible"] == 2
        ),
        2,
        "Probability_Poisonous",
    ),
]

selected_mushrooms = pd.concat(
    selected_parts,
    ignore_index=True,
)

selected_mushrooms["Mushroom_ID"] = [
    f"M{i:02d}"
    for i in range(1, len(selected_mushrooms) + 1)
]

overview_columns = [
    "Mushroom_ID",
    "Story_Group",
    "Original_Index",
    "target",
    "Predicted_Class",
    "Tree_Votes_Edible",
    "Tree_Votes_Poisonous",
    "Vote_Split",
    "Probability_Edible",
    "Probability_Poisonous",
    "odor",
    "gill-size",
    "ring-type",
    "spore-print-color",
    "habitat",
]

selected_mushrooms[overview_columns]


## 8. Export votes and decision paths for the selected mushrooms


In [ ]:
tree_vote_records = []
decision_path_records = []

for _, mushroom in selected_mushrooms.iterrows():
    mushroom_id = mushroom["Mushroom_ID"]
    original_index = int(mushroom["Original_Index"])

    x_one_df = X.loc[[original_index]]
    x_one = x_one_df.values

    for display_order, (
        source_tree_number,
        estimator,
    ) in enumerate(
        zip(SELECTED_TREE_NUMBERS, selected_estimators),
        start=1,
    ):
        tree_name = f"Tree {display_order}"
        tree = estimator.tree_

        vote = estimator.predict(x_one)[0]
        vote_label = class_label(vote)
        tree_probability = estimator.predict_proba(x_one)[0]

        tree_vote_records.append({
            "Mushroom_ID": mushroom_id,
            "Original_Index": original_index,
            "Original_Tree_Number": source_tree_number,
            "Tree_Name": tree_name,
            "Tree_Display_Order": display_order,
            "Is_Front_Tree": int(
                source_tree_number in FRONT_TREE_NUMBERS
            ),
            "Tree_Vote": vote_label,
            "Tree_Probability_Poisonous": tree_probability[
                class_to_col[0]
            ],
            "Tree_Probability_Edible": tree_probability[
                class_to_col[1]
            ],
        })

        node_path = estimator.decision_path(x_one).indices
        leaf_id = estimator.apply(x_one)[0]

        for path_step, node_id in enumerate(node_path):
            is_leaf = tree.children_left[node_id] == -1

            feature_name = ""
            decision_direction = ""
            decision_value = np.nan
            next_node_id = np.nan

            if not is_leaf and path_step < len(node_path) - 1:
                feature_index = tree.feature[node_id]
                feature_name = X.columns[feature_index]
                feature_value = x_one_df.iloc[0, feature_index]

                left_child = tree.children_left[node_id]
                next_node = node_path[path_step + 1]

                decision_direction = (
                    "True" if next_node == left_child else "False"
                )
                decision_value = feature_value
                next_node_id = next_node

            decision_path_records.append({
                "Mushroom_ID": mushroom_id,
                "Original_Index": original_index,
                "Original_Tree_Number": source_tree_number,
                "Tree_Name": tree_name,
                "Tree_Display_Order": display_order,
                "Is_Front_Tree": int(
                    source_tree_number in FRONT_TREE_NUMBERS
                ),
                "Node_ID": node_id,
                "Path_Step": path_step,
                "Feature": feature_name,
                "Is_On_Path": 1,
                "Is_Final_Leaf_For_Mushroom": int(
                    node_id == leaf_id
                ),
                "Tree_Vote": vote_label,
                "Decision_Direction": decision_direction,
                "Decision_Value": decision_value,
                "Next_Node_ID": next_node_id,
            })

df_tree_votes = pd.DataFrame(tree_vote_records)
df_mushroom_paths = pd.DataFrame(decision_path_records)

df_tree_votes.to_csv(
    "tree_votes.csv",
    index=False,
    sep=";",
)

df_mushroom_paths.to_csv(
    "mushroom_decision_paths.csv",
    index=False,
    sep=";",
)

print(f"Exported tree_votes.csv ({len(df_tree_votes):,} rows)")
print(
    "Exported mushroom_decision_paths.csv "
    f"({len(df_mushroom_paths):,} rows)"
)


## 9. Notes

This notebook is intentionally focused on the reproducible modelling and export pipeline used by the visualization.

The Tableau workbook handles the final visual explanation of:

- how individual trees split the data,
- which path a selected mushroom follows,
- how trees vote differently,
- and how those votes combine into a forest-level prediction.

The visualization uses a deliberately small subset of trees for readability; this is a design choice for explanation, not a claim that nine trees are optimal for predictive performance.
